In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

DATA_PATH = 'data/your_dataset.csv'
TRAIN_SIZE = 3510
SEQ_LEN = 256
TEST_SIZE = 100

frame = pd.read_csv(DATA_PATH)
values = frame.to_numpy(dtype=np.float64)
if values.shape[1] < 27:
    raise ValueError('Expected 26 input columns followed by one target column.')
if not np.isfinite(values).all():
    raise ValueError('Dataset contains missing or non-finite values.')
X = values[:, :26]
Y = values[:, 26]
train_end = TRAIN_SIZE + SEQ_LEN
if len(values) < train_end + TEST_SIZE:
    raise ValueError(f'Expected at least {train_end + TEST_SIZE} rows.')


In [ ]:
pipeline = Pipeline([
    ('scale', StandardScaler()),
    ('elasticnet', ElasticNet(max_iter=10000)),
])
selector = GridSearchCV(
    pipeline,
    param_grid={
        'elasticnet__alpha': np.logspace(-5, 1, 20),
        'elasticnet__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
    },
    cv=TimeSeriesSplit(n_splits=5),
    scoring='neg_mean_squared_error',
    n_jobs=-1,
)
selector.fit(X[SEQ_LEN:train_end], Y[SEQ_LEN:train_end])
feature_coefficients = selector.best_estimator_.named_steps['elasticnet'].coef_
print('Best parameters:', selector.best_params_)


In [ ]:
threshold = 1e-5
selected_features = np.flatnonzero(np.abs(feature_coefficients) > threshold)
if selected_features.size == 0:
    raise ValueError('Feature selection removed every input feature.')
print('Selected feature count:', selected_features.size)
print('Selected feature indices:', selected_features)


In [ ]:
X_EN = X[:, selected_features]
data_EN = np.column_stack((X_EN, Y))
selected_frame = pd.DataFrame(data_EN)

# Save only when needed; keep the generated dataset outside version control.
# selected_frame.to_csv('data/selected_dataset.csv', index=False)
